# Blue Team Lab (Network-centric)
## 60-minute investigation using network events (DNS, HTTP, TLS, CONN)

### Scenario
You are a blue team analyst reviewing network telemetry from a short time window.

### Goals
- Identify the most likely compromised host
- Identify suspicious infrastructure (domain/IP)
- Build a timeline of activity
- Recommend containment and one detection improvement
- Document your findings clearly

### Deliverables
1) Answers to the 8 investigation questions
2) A 6–10 sentence incident summary


## Participant Guide (beginner-friendly)

### What you are doing
You are investigating possible malicious activity using **network logs**. Network logs record what computers talked to what destinations, what domains were looked up, and what web requests were made.

### How to run this notebook
1) In Colab: Runtime → Run all.
2) Read the text above each code cell. It tells you what the cell does and what you should see.
3) When you reach the timeline section, you will fill in `HOST_IP = "..."` using a value you copy from the host summary output.

### What a good result looks like
- You can name the internal host (source IP) that looks compromised.
- You can point to suspicious indicators (domain, URI, destination IP).
- You can show a sequence of events (timeline) that supports your conclusion.
- You can recommend containment and a detection improvement.


## Section 1: Load the data

### What
This cell loads the network log data from a CSV file hosted in GitHub.

### Why
Colab does not automatically have files from your computer. Loading from a URL ensures everyone is using the same dataset.

### What to expect
- A small table preview (`df.head(10)`) with columns like `event_type`, `src_ip`, `dst_ip`, `dst_port`.
- Some columns are event-specific:
  - DNS events may use `query`
  - TLS events may use `sni`
  - HTTP events may use `http_host` and `uri`

### If something goes wrong
- If you get an error here, stop. Nothing else will work until the data loads.


In [ ]:
import pandas as pd
import numpy as np

# Colab data source (raw GitHub URL)
NETWORK_CSV_URL = "https://raw.githubusercontent.com/dadirad/blue-team-lab/main/files/network_events.csv"

df = pd.read_csv(NETWORK_CSV_URL)

# Normalize timestamps if present
if "ts" in df.columns:
    df["ts"] = pd.to_datetime(df["ts"], utc=True, errors="coerce")
    df = df.sort_values("ts").reset_index(drop=True)

df.head(10)


## Task 0 (2 minutes): Confirm what data you have

### What
This cell counts the types of network events in the dataset.

### Why
It tells you what evidence exists and what sections will be useful.

### What to expect
You should see event types like:
- DNS (domain lookups)
- TLS (encrypted session metadata, like SNI)
- HTTP (web requests, including URIs)
- CONN (connections, useful for repeated patterns)


In [ ]:
df["event_type"].value_counts(dropna=False)


## Task 1 (8 minutes): Identify the most suspicious host (your investigation target)

### What
This section groups events by internal host (`src_ip`, `src_host`) and summarizes activity.

### Why
In a SIEM, you often start by asking: “Which internal host looks most suspicious during this time window?”

### What to expect
A table where each row is an internal host with metrics like:
- total events
- unique destination IPs
- counts of DNS/HTTP/TLS/CONN events

### What to do
Pick one host that looks suspicious (often the one that shows a suspicious sequence: DNS → HTTP download → repeated connections).
You will copy its IP into `HOST_IP` later.


In [ ]:
def nunique_nonnull(series):
    return series.dropna().nunique()

host_summary = (
    df.groupby(["src_ip", "src_host"], dropna=False)
      .agg(
          first_seen=("ts", "min") if "ts" in df.columns else ("dst_ip", "count"),
          last_seen=("ts", "max") if "ts" in df.columns else ("dst_ip", "count"),
          total_events=("event_type", "count"),
          unique_dst_ips=("dst_ip", "nunique"),
          unique_dns_queries=("query", nunique_nonnull) if "query" in df.columns else ("dst_ip", "nunique"),
          unique_tls_sni=("sni", nunique_nonnull) if "sni" in df.columns else ("dst_ip", "nunique"),
          unique_http_hosts=("http_host", nunique_nonnull) if "http_host" in df.columns else ("dst_ip", "nunique"),
          http_events=("event_type", lambda s: (s=="HTTP").sum()),
          dns_events=("event_type", lambda s: (s=="DNS").sum()),
          tls_events=("event_type", lambda s: (s=="TLS").sum()),
          conn_events=("event_type", lambda s: (s=="CONN").sum())
      )
      .reset_index()
      .sort_values(["total_events", "unique_dst_ips"], ascending=False)
)

host_summary


## Task 2 (10 minutes): Find suspicious domains and suspicious HTTP paths

### What
This task looks for two common attacker signals:
1) **Rare DNS queries** (domains that appear only once)
2) **Suspicious HTTP URIs** (paths that look like downloads or payloads)

### Why
- Malicious domains often have low prevalence in a small time window.
- Payload delivery often appears as an HTTP request to a suspicious URI (like `/payload.exe`).

### What to expect
- A list of rare domains (if DNS `query` exists)
- A table of suspicious HTTP rows (if `uri` exists)

### What to do
Capture 1–2 indicators you think are suspicious (domain + URI + timestamp).


In [ ]:
if "query" in df.columns:
    dns = df[df["event_type"]=="DNS"].copy()
    dns_counts = dns["query"].dropna().value_counts()
    rare_dns = dns_counts[dns_counts == 1]
    display(rare_dns.head(25))
else:
    print("No 'query' column found. Skipping rare DNS analysis.")


In [ ]:
if "uri" in df.columns and "http_host" in df.columns:
    http = df[df["event_type"]=="HTTP"].copy()
    pattern = r"(payload|\.exe|\.ps1|\.dll|\.bin|\.dat|/download|/update|/install)"
    susp_http = http[http["uri"].fillna("").str.contains(pattern, case=False, regex=True)]
    cols = [c for c in ["ts","src_ip","src_host","dst_ip","dst_port","http_host","uri","method","status","bytes_in","notes"] if c in df.columns]
    display(susp_http[cols].sort_values("ts") if "ts" in df.columns else susp_http[cols])
else:
    print("No HTTP columns (uri/http_host) found. Skipping suspicious HTTP analysis.")


## Task 3 (10 minutes): Look for repeated connections (beaconing hint)

### What
This task checks for repeated CONN events from the same host to the same destination.

### Why
Some malware "beacons" by connecting out repeatedly to a command-and-control destination at regular intervals.

### What to expect
A table of candidate flows showing:
- number of connections (`count`)
- average time between connections (`mean_delta`)
- consistency (`std_delta`)

### What to do
Note any destination with multiple connections and relatively consistent timing.


In [ ]:
required = {"src_ip","dst_ip","dst_port","event_type"}
if required.issubset(df.columns) and "ts" in df.columns:
    conn = df[df["event_type"]=="CONN"].copy()
    conn = conn.sort_values(["src_ip","dst_ip","dst_port","ts"])
    conn["delta_sec"] = conn.groupby(["src_ip","dst_ip","dst_port"])["ts"].diff().dt.total_seconds()

    group_cols = [c for c in ["src_ip","src_host","dst_ip","dst_port"] if c in conn.columns]
    beacon_stats = (
        conn.groupby(group_cols, dropna=False)
            .agg(
                count=("ts","count"),
                mean_delta=("delta_sec","mean"),
                std_delta=("delta_sec","std"),
                first=("ts","min"),
                last=("ts","max")
            )
            .reset_index()
    )

    beacon_candidates = beacon_stats[(beacon_stats["count"]>=4) & (beacon_stats["mean_delta"].notna())].copy()
    beacon_candidates = beacon_candidates.sort_values(["count","std_delta"], ascending=[False, True])
    display(beacon_candidates.head(25))
else:
    print("Beaconing analysis requires src_ip, dst_ip, dst_port, event_type, and ts columns. Skipping.")


## Task 4 (15 minutes): Build the timeline for your chosen host

### What
This section filters all events for ONE host and shows them in time order.

### Why
A timeline is how analysts explain "what happened" with evidence (not guesses).

### What to do
1) Copy the suspicious host IP from the Task 1 host summary (example: 10.0.5.23)
2) Paste it into `HOST_IP = "..."`
3) Run the code cell again

### What to expect
A typical suspicious pattern might look like:
- DNS query to a suspicious domain
- TLS SNI or HTTP host matching that domain
- HTTP request to a landing page and/or payload
- follow-on destination(s)
- repeated CONN events (beaconing hint)


In [ ]:
HOST_IP = ""  # e.g., "10.0.5.23"
if not HOST_IP:
    print("Set HOST_IP to a value like '10.0.5.23' and re-run this cell.")
else:
    host_events = df[df["src_ip"]==HOST_IP].copy()
    cols = [c for c in ["ts","event_type","src_ip","src_host","dst_ip","dst_port","query","sni","http_host","uri","status","bytes_in","notes"] if c in df.columns]
    display(host_events[cols].sort_values("ts") if "ts" in df.columns else host_events[cols])


## Optional pivot: Focus on one suspicious domain

### What
This pivot shows all rows where a domain appears in DNS (`query`), TLS (`sni`), or HTTP (`http_host`).

### Why
It helps confirm the indicator and check whether more than one host is involved.

### What to do
If you identified a suspicious domain in Task 2, paste it into `SUSP_DOMAIN` and run the next cell.


In [ ]:
SUSP_DOMAIN = ""  # optional: set to a suspicious domain and re-run
if not SUSP_DOMAIN:
    print("Optional: set SUSP_DOMAIN to pivot across DNS/TLS/HTTP and re-run.")
else:
    mask = False
    if "query" in df.columns:
        mask = mask | (df["query"]==SUSP_DOMAIN)
    if "sni" in df.columns:
        mask = mask | (df["sni"]==SUSP_DOMAIN)
    if "http_host" in df.columns:
        mask = mask | (df["http_host"]==SUSP_DOMAIN)

    domain_events = df[mask].copy()
    cols = [c for c in ["ts","event_type","src_ip","src_host","dst_ip","dst_port","query","sni","http_host","uri","status","bytes_in","notes"] if c in df.columns]
    display(domain_events[cols].sort_values("ts") if "ts" in df.columns else domain_events[cols])


## Submit: 8 Investigation Questions

### What
This is where you document your findings.

### Why
In real incident response, the write-up is how others take action: containment, remediation, and detection updates.

### What to do
Answer each question with evidence (timestamps, host IP, domains, URI paths, destination IPs).

1) Suspected compromised host (IP + hostname):
2) Most suspicious external destination (domain or IP):
3) First suspicious timestamp (with evidence):
4) Primary lead (what triggered your investigation?):
5) Evidence that supports malicious activity (2–3 bullets):
6) Best-guess attack technique (plain language is fine):
7) Immediate containment actions (1–2 actions):
8) One detection improvement idea:

## Incident Summary (6–10 sentences)
**Title:** Suspected malicious activity on `<host>`

- At `<time>`, host `<internal IP>` initiated `<protocol>` activity to `<domain/IP>`.
- Evidence includes `<DNS query / TLS SNI / HTTP URI>` and supporting connection patterns.
- The sequence suggests `<download / beaconing>` within the capture window.
- Follow-on traffic to `<second infrastructure>` indicates possible command and control.
- Confidence: `<high/medium/low>` because `<reason>`.
- Recommended containment: `<actions>`.
- Detection improvement: `<specific rule/tuning idea>`.
